In [1]:
import sys
import argparse
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import numpy as np
import anndata
import statsmodels.api as sm
import statsmodels.formula.api as smf
import tqdm
import pandas as pd



import anndata
import csv
import gzip
import os
import scipy.io
import numpy as np 
import anndata

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import scanpy
from matplotlib.pyplot import rc_context




In [2]:

# --- INIT ---

# which data to use
head_folder = "/".join(os.getcwd().split("/")[:3])
data_head_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint' % head_folder
hvg_to_use = 'prior_knowledge_guided'

# get corresponding paths
adata_path = '%s/adata_qc_filtered.h5ad' % data_head_folder
hvg_folder = '%s/data_annotations/hvg/%s' % (data_head_folder,hvg_to_use)
scvi_res_folder = '%s/scvi_res' % hvg_folder
scvi_mean_embedding_df_path = '%s/scvi_mean_embedding.tsv' % scvi_res_folder
umap_path_out = '%s/scvi_mean_umap_embedding.tsv' % scvi_res_folder
cluster_resolution=0.05
subcluster_resolution = 0.2


# SMC subclusters
clustering_folder = '%s/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/%s/scvi_res/clustering/res_%s/subclustering/res_%s' % (head_folder,hvg_to_use,cluster_resolution,subcluster_resolution)
cluster_df_fileout = '%s/smc_subclusters.tsv' % (clustering_folder)



# settings
min_prop = 1e-7




In [3]:
# --- LOAD ADATA ---

adata = anndata.read_h5ad(adata_path)

adata


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [4]:
# --- ADD EMBEDDINGS TO ADATA ---



# ** load clusters **
cluster_df = pd.read_table(cluster_df_fileout,sep='\t',index_col='index')

# ** make sure everything in the same order
adata = adata[list(cluster_df.index)]


# ** add embeddings **
adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])

adata


/var/folders/y2/0zd17m8j6bz5j5xv0cy2dw680000gp/T/ipykernel_9314/564461275.py:13: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cluster"] = np.array(cluster_df['smc_subcluster'])


AnnData object with n_obs × n_vars = 145271 × 32285
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2'

In [5]:
# --- GET THE UNIQUE DESCRIPTION AND CLUSTERS ---

descriptions = list(adata.obs['description'].unique())
clusters = sorted(list(adata.obs['cluster'].unique()))
clusters = list(filter(lambda x: ~np.isnan(x),clusters)) # get rid of NaN cluster (only relevant for SMC subcluster)
clusters = list(map(lambda x: int(x),clusters))


print("Unique descriptions:\n",descriptions)
print("Unique clusters:\n",clusters)


# --- LIMIT ADATA TO GENES THAT MEET MINIMUM PSEUDOBULK COUNT THRESHOLD ---


# ** get gene pseudobulk counts
adata.var['pseudobulk_count'] = np.array(np.sum(adata.X,axis=0)).flatten()

# ** get the number of cells in the smallest cell type
smallest_cell_type_adata = adata[adata.obs['cluster'] == np.max(clusters)]

# ** get pseudobulk cutoff **
pseudobulk_threshold = min_prop * np.sum(smallest_cell_type_adata.obs['lib_size'])

# ** limit adata to this **
adata = adata[:,adata.var['pseudobulk_count'] >= pseudobulk_threshold]


adata


Unique descriptions:
 ['male aligned bmal1-ko', 'male misaligned bmal1-control', 'female aligned bmal1-ko', 'female misaligned bmal1-control', 'male aligned bmal1-control', 'female aligned bmal1-control']
Unique clusters:
 [0, 1, 2, 3, 4, 6, 7]


View of AnnData object with n_obs × n_vars = 145271 × 26120
    obs: 'sample_id', 'within_experiment_batch_id', 'lib_size', 'sex', 'misaligned', 'zt', 'bmal1_ko', 'batch', 'percent_mito', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mito_qc_pass', 'umi_qc_pass', 'doublet_qc_pass', 'doublet_score', 'qc_pass', 'concat_batch', 'description', 'cluster'
    var: 'ensembl_id', 'n_cells_by_counts-0', 'mean_counts-0', 'log1p_mean_counts-0', 'pct_dropout_by_counts-0', 'total_counts-0', 'log1p_total_counts-0', 'n_cells_by_counts-1', 'mean_counts-1', 'log1p_mean_counts-1', 'pct_dropout_by_counts-1', 'total_counts-1', 'log1p_total_counts-1', 'n_cells_by_counts-2', 'mean_counts-2', 'log1p_mean_counts-2', 'pct_dropout_by_counts-2', 'total_counts-2', 'log1p_total_counts-2', 'pseudobulk_count'

In [6]:
# --- LIMIT CLUSTERS ---

clusters = clusters[:-2]
clusters


[0, 1, 2, 3, 4]

In [7]:
# --- IDENTIFY GENES MEETING THE MINIMUM PROP IN EACH CLUSTER / CONDITION COMBO ---

import warnings
warnings.filterwarnings('ignore')

genes_to_est = set()
for i, description in enumerate(descriptions):
    for j, cluster in enumerate(clusters):
        print("Description: %s; Cluster: %s" % (i,j))
            
        cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
        cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten()  / np.sum(cluster_condition_adata.obs['lib_size'])
        cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] >= min_prop]
        genes_to_est.update(list(cluster_condition_adata.var_names))

# make it a list
genes_to_est = list(genes_to_est)



Description: 0; Cluster: 0
Description: 0; Cluster: 1
Description: 0; Cluster: 2
Description: 0; Cluster: 3
Description: 0; Cluster: 4
Description: 1; Cluster: 0
Description: 1; Cluster: 1
Description: 1; Cluster: 2
Description: 1; Cluster: 3
Description: 1; Cluster: 4
Description: 2; Cluster: 0
Description: 2; Cluster: 1
Description: 2; Cluster: 2
Description: 2; Cluster: 3
Description: 2; Cluster: 4
Description: 3; Cluster: 0
Description: 3; Cluster: 1
Description: 3; Cluster: 2
Description: 3; Cluster: 3
Description: 3; Cluster: 4
Description: 4; Cluster: 0
Description: 4; Cluster: 1
Description: 4; Cluster: 2
Description: 4; Cluster: 3
Description: 4; Cluster: 4
Description: 5; Cluster: 0
Description: 5; Cluster: 1
Description: 5; Cluster: 2
Description: 5; Cluster: 3
Description: 5; Cluster: 4


In [8]:
len(genes_to_est)

20159

In [9]:
clustering_folder

'/Users/mingyaolab/Dropbox/aorta_circadian_data/datasets/joint/data_annotations/hvg/prior_knowledge_guided/scvi_res/clustering/res_0.05/subclustering/res_0.2'

In [7]:
# --- MAKE THE FOLDER OUTS ---

# head folder
reg_head_folder = '%s/nonparametric_reg' % clustering_folder
if not os.path.exists(reg_head_folder):
    os.makedirs(reg_head_folder)


# subfolders for each cluster / condition combo
cluster_description_folder_out_dict = {}
for cluster in clusters:
    cluster_reg_folder = '%s/cluster_%s' % (reg_head_folder,cluster)
    if not os.path.exists(cluster_reg_folder):
        os.makedirs(cluster_reg_folder)
    for description in descriptions:
        cluster_description_reg_folder = '%s/%s' % (cluster_reg_folder,description)
        if not os.path.exists(cluster_description_reg_folder):
            os.makedirs(cluster_description_reg_folder)
            
        # update dict
        if cluster not in cluster_description_folder_out_dict:
            cluster_description_folder_out_dict[cluster] = {}
        cluster_description_folder_out_dict[cluster][description] = cluster_description_reg_folder

    
    

In [8]:
# --- WRITE OUT THE GENES TO ESTIMATE ---

genes_to_est_fileout = '%s/genes_to_est.txt' % reg_head_folder
with open(genes_to_est_fileout,"wb") as file_obj:
    file_obj.write("\n".join(genes_to_est).encode())




In [9]:
# --- ADD PHASE COL ---

adata.obs['phase'] = np.array((adata.obs['zt'] / 24.0) * 2 * np.pi)



In [ ]:
# --- FIT ---


import torch
import tempo2
from tempo2 import identify_de_novo_cyclers

failed_list = []
for cluster in clusters:
    for description in descriptions:
        
        try:
        
            print("----------------")
            print("Cluster %s; Description %s" % (cluster,description))


            # ** get cluster, condition adata **
            cluster_condition_adata = adata[(adata.obs['cluster'] == cluster) & (adata.obs['description'] == description)]
            cluster_condition_adata = cluster_condition_adata[:,genes_to_est]
            print(cluster_condition_adata.shape)

            
            # ** prep **
            cluster_condition_adata.var['prop'] = np.array(np.sum(cluster_condition_adata.X,axis=0)).flatten() / np.sum(cluster_condition_adata.obs['lib_size'])
            if 'log_L' not in adata.obs:
                cluster_condition_adata.obs['log_L'] = np.array(np.log(cluster_condition_adata.obs['lib_size']))
            cluster_condition_adata = cluster_condition_adata[:,cluster_condition_adata.var['prop'] > 0]

            
            
            # ** run **
            tempo2.identify_de_novo_cyclers.run(adata = cluster_condition_adata,
                folder_out = cluster_description_folder_out_dict[cluster][description],
                num_grid_points = 4,
                phases_sampled = torch.Tensor(np.array(cluster_condition_adata.obs['phase'])).unsqueeze(1), # torch.Tensor(np.array(adata.obs['phase'])).unsqueeze(1),
                cell_phase_dist = None, # cell_posterior_obj, # cell_posterior_obj # cell_posterior_obj
                use_nb = True,
                log_mean_log_disp_coef= None,# None, torch.Tensor(np.array([-3.001958370208740234e+00, -1.134198158979415894e-01]))
                lr = 1e-1,
                num_waveform_est_cell_samples = 1,
                num_waveform_est_gene_samples = 1,
                num_bf_est_cell_samples = 5,
                num_bf_est_gene_samples = 5,
                vi_max_epochs = 300,
                vi_print_epoch_loss = True,
                vi_improvement_window = 5,
                vi_convergence_criterion = 1e-3,
                cosinor_zero_frac_num_samples_per_waveform = 100,
                num_waveform_bf_cell_samples = 5,
                num_waveform_bf_gene_samples = 5,
                )


            
             
            
        except Exception as e:
            print(e)
            
            failed = '%s_%s' % (cluster,description)
            failed_list.append(failed)
            
            
            
            


----------------
Cluster 0; Description male aligned bmal1-ko
(11114, 20993)
Bin 0 size: 50
Bin 1 size: 50
Bin 2 size: 50
Bin 3 size: 50
Bin 4 size: 50
Bin 5 size: 50
Bin 6 size: 50
Bin 7 size: 50
Bin 8 size: 50
Bin 9 size: 50
Bin 10 size: 50
Bin 11 size: 50
Bin 12 size: 50
Bin 13 size: 50
Bin 14 size: 50
Bin 15 size: 50
Bin 16 size: 50
Bin 17 size: 50
Bin 18 size: 50
Bin 19 size: 50
Bin 20 size: 50
Bin 21 size: 21
Bin 22 size: 8
Bin 23 size: 7
Bin 24 size: 4
Bin 25 size: 4
Bin 26 size: 1
Bin 27 size: 0
Bin 28 size: 0


100%|███████████████████████████████████████| 1095/1095 [00:34<00:00, 31.68it/s]


Iter: 0; ELBO 8101.346: E[LL]: -8101.346; KL: 0.0
Iter: 1; ELBO 7904.204: E[LL]: -7904.144; KL: 0.059
Iter: 2; ELBO 7755.937: E[LL]: -7755.745; KL: 0.192
Iter: 3; ELBO 7630.845: E[LL]: -7630.487; KL: 0.358
Iter: 4; ELBO 7516.28: E[LL]: -7515.745; KL: 0.534
Iter: 5; ELBO 7406.029: E[LL]: -7405.317; KL: 0.712
Iter: 6; ELBO 7302.049: E[LL]: -7301.163; KL: 0.886
Iter: 7; ELBO 7204.163: E[LL]: -7203.109; KL: 1.053
Iter: 8; ELBO 7107.354: E[LL]: -7106.143; KL: 1.212
Iter: 9; ELBO 7015.475: E[LL]: -7014.114; KL: 1.361
Iter: 10; ELBO 6924.939: E[LL]: -6923.436; KL: 1.504
Improvement: 0.07989497106659871
Iter: 11; ELBO 6839.66: E[LL]: -6838.019; KL: 1.641
Improvement: 0.07550409510106382
Iter: 12; ELBO 6759.368: E[LL]: -6757.591; KL: 1.777
Improvement: 0.07272685040726412
Iter: 13; ELBO 6679.996: E[LL]: -6678.079; KL: 1.917
Improvement: 0.07070741726660013
Iter: 14; ELBO 6606.87: E[LL]: -6604.804; KL: 2.065
Improvement: 0.06880778106842778
Iter: 15; ELBO 6537.973: E[LL]: -6535.745; KL: 2.228
Im

100%|███████████████████████████████████████| 1095/1095 [00:28<00:00, 37.83it/s]


Iter: 0; ELBO 6880.467: E[LL]: -6880.467; KL: 0.0
Iter: 1; ELBO 6696.908: E[LL]: -6696.851; KL: 0.058
Iter: 2; ELBO 6560.101: E[LL]: -6559.91; KL: 0.19
Iter: 3; ELBO 6447.374: E[LL]: -6447.016; KL: 0.358
Iter: 4; ELBO 6346.067: E[LL]: -6345.528; KL: 0.538
Iter: 5; ELBO 6250.42: E[LL]: -6249.701; KL: 0.72
Iter: 6; ELBO 6160.561: E[LL]: -6159.663; KL: 0.898
Iter: 7; ELBO 6073.215: E[LL]: -6072.146; KL: 1.07
Iter: 8; ELBO 5991.244: E[LL]: -5990.012; KL: 1.233
Iter: 9; ELBO 5912.438: E[LL]: -5911.049; KL: 1.388
Iter: 10; ELBO 5834.28: E[LL]: -5832.744; KL: 1.536
Improvement: 0.08338015963036094
Iter: 11; ELBO 5759.37: E[LL]: -5757.69; KL: 1.68
Improvement: 0.078169634850487
Iter: 12; ELBO 5687.121: E[LL]: -5685.298; KL: 1.822
Improvement: 0.07502498588051987
Iter: 13; ELBO 5618.033: E[LL]: -5616.064; KL: 1.968
Improvement: 0.07275375588808064
Iter: 14; ELBO 5552.66: E[LL]: -5550.538; KL: 2.122
Improvement: 0.07088955065735769
Iter: 15; ELBO 5491.57: E[LL]: -5489.28; KL: 2.29
Improvement: 0

100%|███████████████████████████████████████| 1094/1094 [00:29<00:00, 37.63it/s]


Iter: 0; ELBO 6069.476: E[LL]: -6069.476; KL: 0.0
Iter: 1; ELBO 5920.024: E[LL]: -5919.967; KL: 0.058
Iter: 2; ELBO 5805.775: E[LL]: -5805.585; KL: 0.191
Iter: 3; ELBO 5712.098: E[LL]: -5711.738; KL: 0.36
Iter: 4; ELBO 5627.249: E[LL]: -5626.707; KL: 0.542
Iter: 5; ELBO 5548.443: E[LL]: -5547.717; KL: 0.726
Iter: 6; ELBO 5473.157: E[LL]: -5472.25; KL: 0.906
Iter: 7; ELBO 5401.665: E[LL]: -5400.586; KL: 1.08
Iter: 8; ELBO 5333.155: E[LL]: -5331.911; KL: 1.244
Iter: 9; ELBO 5266.348: E[LL]: -5264.948; KL: 1.4
Iter: 10; ELBO 5202.217: E[LL]: -5200.669; KL: 1.547
Improvement: 0.07827348719802496
Iter: 11; ELBO 5141.403: E[LL]: -5139.713; KL: 1.69
Improvement: 0.07334403833733405
Iter: 12; ELBO 5082.637: E[LL]: -5080.807; KL: 1.83
Improvement: 0.07019636454696576
Iter: 13; ELBO 5025.649: E[LL]: -5023.675; KL: 1.974
Improvement: 0.06794886300237268
Iter: 14; ELBO 4973.131: E[LL]: -4971.005; KL: 2.125
Improvement: 0.06600702669416036
Iter: 15; ELBO 4924.091: E[LL]: -4921.801; KL: 2.29
Improve